# Transformer Foundations, Part 5B: Encoder-Decoder PyTorch Lab

Part 5A proved why a mean summary cannot support position-specific retrieval. This lab keeps the source memory addressable and trains one target writer on the same four-digit reversal diagnostic.

| Build step | Contract |
|---|---|
| 1 | source examples and shifted target prefixes |
| 2 | encoder memory `(B,S,D)` |
| 3 | `CrossAttention`: decoder Q, encoder K/V |
| 4 | target logits `(B,T,V)` with `T != S` |
| 5 | teacher-forced and free-running evaluation |
| 6 | learned routing map versus the predicted anti-diagonal |

This diagnostic is deliberately synthetic. It proves routing mechanics, not translation quality.


In [ ]:
# Imports and deterministic lab state
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

SEED = 505
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device("cpu")

PAD_ID, BOS_ID, EOS_ID = 0, 1, 2
DIGIT_OFFSET = 3
DIGIT_COUNT = 8
VOCAB_SIZE = DIGIT_OFFSET + DIGIT_COUNT
SOURCE_LENGTH = 4
TARGET_LENGTH = SOURCE_LENGTH + 1


## 1. Build the source once; shift only the target

A source example has four digit tokens. The target writer receives `[BOS] + reversed_source`; it is graded against `reversed_source + [EOS]`.

Source length is `4`; target length is `5`. The mismatch is intentional evidence that cross-attention's score grid is rectangular.


In [ ]:
# Deterministic train/validation reversal data and DataLoaders
generator = torch.Generator().manual_seed(SEED)
all_sources = torch.randint(
    DIGIT_OFFSET,
    VOCAB_SIZE,
    (1280, SOURCE_LENGTH),
    generator=generator,
)
reversed_sources = all_sources.flip(dims=[1])
decoder_inputs = torch.cat(
    [torch.full((len(all_sources), 1), BOS_ID), reversed_sources], dim=1
)
decoder_targets = torch.cat(
    [reversed_sources, torch.full((len(all_sources), 1), EOS_ID)], dim=1
)

train_data = TensorDataset(
    all_sources[:1024], decoder_inputs[:1024], decoder_targets[:1024]
)
validation_data = TensorDataset(
    all_sources[1024:], decoder_inputs[1024:], decoder_targets[1024:]
)
train_loader = DataLoader(
    train_data,
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
validation_loader = DataLoader(validation_data, batch_size=64, shuffle=False)

source_ids, target_in, target_out = next(iter(train_loader))
print("source:", tuple(source_ids.shape))
print("decoder input:", tuple(target_in.shape))
print("decoder target:", tuple(target_out.shape))
assert source_ids.size(1) != target_in.size(1)


## 2. CrossAttention makes tensor roles explicit

The class below is intentionally separate from decoder self-attention:

```text
decoder states -> query projection
encoder memory -> key projection and value projection
scores          -> (batch, heads, target, source)
```

A source padding mask would block invalid source columns. This fixed-length diagnostic has no padding, so the argument remains `None`.


In [ ]:
# Q from decoder; K and V from encoder memory
class CrossAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        if d_model % n_heads:
            raise ValueError("d_model must be divisible by n_heads")
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def split_heads(self, value):
        batch, length, width = value.shape
        return value.view(
            batch, length, self.n_heads, self.head_dim
        ).transpose(1, 2)

    def forward(self, decoder_states, encoder_memory, source_padding_mask=None):
        q = self.split_heads(self.q_proj(decoder_states))
        k = self.split_heads(self.k_proj(encoder_memory))
        v = self.split_heads(self.v_proj(encoder_memory))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        if source_padding_mask is not None:
            scores = scores.masked_fill(
                source_padding_mask[:, None, None, :], float("-inf")
            )
        weights = torch.softmax(scores, dim=-1)
        mixed = weights @ v
        batch, _, target_length, _ = mixed.shape
        mixed = mixed.transpose(1, 2).contiguous().view(
            batch, target_length, -1
        )
        return self.out_proj(mixed), weights


In [ ]:
# One decoder block: causal target self-attention, then source retrieval
class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.self_norm = nn.LayerNorm(d_model)
        self.self_attention = nn.MultiheadAttention(
            d_model, n_heads, batch_first=True
        )
        self.cross_norm = nn.LayerNorm(d_model)
        self.cross_attention = CrossAttention(d_model, n_heads)
        self.ffn_norm = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x, memory):
        target_length = x.size(1)
        causal_mask = torch.triu(
            torch.ones(target_length, target_length, dtype=torch.bool, device=x.device),
            diagonal=1,
        )
        normalized = self.self_norm(x)
        self_update, _ = self.self_attention(
            normalized, normalized, normalized, attn_mask=causal_mask
        )
        x = x + self_update
        cross_update, weights = self.cross_attention(
            self.cross_norm(x), memory
        )
        x = x + cross_update
        x = x + self.ffn(self.ffn_norm(x))
        return x, weights


In [ ]:
# A compact encoder-decoder with addressable source memory
class ReversalTransformer(nn.Module):
    def __init__(self, d_model=48, n_heads=4):
        super().__init__()
        self.token_embedding = nn.Embedding(VOCAB_SIZE, d_model)
        self.source_position = nn.Embedding(SOURCE_LENGTH, d_model)
        self.target_position = nn.Embedding(TARGET_LENGTH, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=96,
            dropout=0.0,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=1, enable_nested_tensor=False
        )
        self.decoder = DecoderBlock(d_model, n_heads, 96)
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, VOCAB_SIZE)

    def encode(self, source_ids):
        positions = torch.arange(source_ids.size(1), device=source_ids.device)
        return self.encoder(
            self.token_embedding(source_ids) + self.source_position(positions)
        )

    def decode(self, decoder_ids, memory):
        positions = torch.arange(decoder_ids.size(1), device=decoder_ids.device)
        states = self.token_embedding(decoder_ids) + self.target_position(positions)
        states, weights = self.decoder(states, memory)
        return self.lm_head(self.final_norm(states)), weights

    def forward(self, source_ids, decoder_ids):
        memory = self.encode(source_ids)
        logits, weights = self.decode(decoder_ids, memory)
        return logits, weights, memory

model = ReversalTransformer().to(DEVICE)
logits, weights, memory = model(source_ids, target_in)
print("encoder memory:", tuple(memory.shape))
print("target logits:", tuple(logits.shape))
print("cross-attention:", tuple(weights.shape))
assert weights.shape[-2:] == (TARGET_LENGTH, SOURCE_LENGTH)


#### Walkthrough: the boundary is now visible

The encoder creates one vector per source position. The decoder creates one state per target-prefix position. `CrossAttention` is the only place those workspaces meet, so a shape error cannot silently blur their roles.

## 3. Train with gold prefixes; evaluate both conditions

The loss below is teacher forced because every decoder input contains the correct earlier target tokens. After training, free-running decoding starts with only BOS and feeds predictions back.


In [ ]:
# Manual teacher-forced optimization loop
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-3, weight_decay=0.01)
history = []
model.train()
for epoch in range(24):
    epoch_loss = 0.0
    for source_batch, input_batch, target_batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        batch_logits, _, _ = model(source_batch, input_batch)
        loss = F.cross_entropy(
            batch_logits.flatten(0, 1), target_batch.flatten()
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
    history.append(epoch_loss / len(train_loader))

print(f"loss: {history[0]:.4f} -> {history[-1]:.4f}")
assert history[-1] < 0.15
plt.figure(figsize=(8, 3))
plt.plot(history)
plt.xlabel("epoch")
plt.ylabel("teacher-forced loss")
plt.title("Reversal training history")
plt.tight_layout()
plt.show()


In [ ]:
# Teacher-forced token accuracy on held-out sequences
model.eval()
correct = total = 0
with torch.no_grad():
    for source_batch, input_batch, target_batch in validation_loader:
        batch_logits, _, _ = model(source_batch, input_batch)
        predictions = batch_logits.argmax(dim=-1)
        correct += int(predictions.eq(target_batch).sum())
        total += target_batch.numel()
teacher_forced_accuracy = correct / total
print(f"teacher-forced token accuracy: {teacher_forced_accuracy:.3f}")
assert teacher_forced_accuracy > 0.97


In [ ]:
# Free-running greedy decoding: encode once, then reuse source memory
@torch.no_grad()
def greedy_decode(source_batch):
    model.eval()
    memory = model.encode(source_batch)
    generated = torch.full(
        (source_batch.size(0), 1), BOS_ID, dtype=torch.long
    )
    for _ in range(TARGET_LENGTH):
        logits, _ = model.decode(generated, memory)
        next_ids = logits[:, -1].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_ids], dim=1)
    return generated[:, 1:]

validation_sources = all_sources[1024:]
expected = decoder_targets[1024:]
generated = greedy_decode(validation_sources)
token_accuracy = generated.eq(expected).float().mean().item()
sequence_accuracy = generated.eq(expected).all(dim=1).float().mean().item()
print(f"free-running token accuracy:    {token_accuracy:.3f}")
print(f"free-running sequence accuracy: {sequence_accuracy:.3f}")
assert sequence_accuracy > 0.90


## 4. Did the learned route match the prediction?

The first four decoder rows predict the four reversed digits. Their expected source addresses are `3, 2, 1, 0`. EOS has no single required source position, so it is excluded from the routing score.

Attention is a diagnostic, not a complete explanation: values, residual streams, and the FFN also affect logits.


In [ ]:
# Held-out learned routing map, averaged across heads
example_source = validation_sources[:1]
example_input = decoder_inputs[1024:1025]
with torch.no_grad():
    example_logits, example_weights, _ = model(example_source, example_input)
mean_weights = example_weights.mean(dim=1)[0].cpu().numpy()
digit_rows = mean_weights[:SOURCE_LENGTH]
routed_positions = digit_rows.argmax(axis=-1)
expected_positions = np.arange(SOURCE_LENGTH - 1, -1, -1)

fig, ax = plt.subplots(figsize=(6, 4))
image = ax.imshow(mean_weights, vmin=0, vmax=1, cmap="magma")
ax.set_xticks(range(SOURCE_LENGTH), example_source[0].tolist())
ax.set_yticks(range(TARGET_LENGTH), ["digit 0", "digit 1", "digit 2", "digit 3", "EOS"])
ax.set_xlabel("source position/token")
ax.set_ylabel("decoder query")
ax.set_title("Learned cross-attention routing")
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()

routing_accuracy = np.mean(routed_positions == expected_positions)
print("argmax source addresses:", routed_positions.tolist())
print(f"routing accuracy: {routing_accuracy:.2f}")
assert routing_accuracy >= 0.75


## Part 5B scorecard

| Question | Measured answer |
|---|---|
| Can source and target lengths differ? | Cross-attention produced `(target=5, source=4)` rows |
| Did teacher forcing learn the task? | Held-out token accuracy printed above |
| Did free-running evaluation survive its own history? | Held-out sequence accuracy printed above |
| Did routing resemble reversal? | Cross-attention argmaxes were compared with `3,2,1,0` |

**Your turn:** change only `SOURCE_LENGTH` from `4` to `5`, then predict the new cross-attention shape before rerunning.

**Toy-to-real bridge:** T5 and BART widen and stack the same three paths: bidirectional source self-attention, causal target self-attention, and target-to-source cross-attention. Production inference caches encoder memory and layer-specific cross-KV once, while target self-KV grows.

**Next boundary:** Part 6 returns to decoder-only models and modernizes the block without reteaching this source/target split.
